In [10]:
import polars as pl
import os

from pathlib import Path

In [11]:
DONOR = 'donor_4'
NUM_ITEMS = 10000

In [12]:
WORKING_PATH = Path('/group/pmc021/amunif/epi-thesis/workflow/16_Pairwise Ranking Healthy Liver/')
DATASET_PATH = WORKING_PATH / 'dataset'/ DONOR
OUTPUT_PATH  = WORKING_PATH / 'output' / str(NUM_ITEMS) / DONOR

In [13]:
### Merge the results into single CSV file

# Read all CSV into single dataframe
pl_df = pl.read_csv(OUTPUT_PATH / 'test' / "*.csv")

In [14]:
pl_df

seed,histone_marker,epochs_trained,train_loss_avg,train_accuracy_avg,train_auc_avg,val_loss_avg,val_accuracy_avg,val_auc_avg,test_accuracy,test_auc
i64,str,i64,f64,f64,f64,f64,f64,f64,f64,f64
1011,"""H3K4me3""",62,0.5802,72.364,0.7244,0.5973,71.2065,0.7139,71.6,0.7158
123,"""H3K4me3""",39,0.5896,71.6259,0.7172,0.5486,73.7974,0.739,71.6,0.7169
42,"""H3K4me3""",78,0.5824,71.9846,0.7195,0.562,73.1551,0.7356,71.8,0.7205
456,"""H3K4me3""",100,0.5694,73.0338,0.7298,0.5501,73.724,0.7402,70.7,0.7087
789,"""H3K4me3""",41,0.5862,72.0654,0.7212,0.5835,70.4073,0.7061,72.4,0.7236
…,…,…,…,…,…,…,…,…,…,…
1011,"""H3K4me3-H3K9ac-H3K9me3-H3K27ac…",54,0.5571,74.0319,0.7402,0.5833,71.1722,0.7102,73.6,0.7359
123,"""H3K4me3-H3K9ac-H3K9me3-H3K27ac…",43,0.5544,74.0214,0.7402,0.52,75.4395,0.7521,74.1,0.7415
42,"""H3K4me3-H3K9ac-H3K9me3-H3K27ac…",95,0.5457,74.4341,0.7438,0.5275,76.2684,0.7648,73.8,0.7392


In [15]:
summary_df = (
    pl_df
    .group_by("histone_marker")
    .agg([
        pl.col("val_accuracy_avg").mean().alias("val_accuracy_mean"),
        pl.col("val_accuracy_avg").std().alias("val_accuracy_std"),
        pl.col("test_accuracy").mean().alias("test_accuracy_mean"),
        pl.col("test_accuracy").std().alias("test_accuracy_std"),
    ])
    .sort("test_accuracy_mean", descending=True)
)

In [16]:
summary_df

histone_marker,val_accuracy_mean,val_accuracy_std,test_accuracy_mean,test_accuracy_std
str,f64,f64,f64,f64
"""H3K9ac-H3K9me3-H3K27ac-H3K27me…",73.6559,1.38498,74.08,1.080278
"""H3K9ac-H3K27ac-H3K27me3""",73.52056,1.326337,74.02,0.779102
"""H3K4me3-H3K9ac-H3K9me3-H3K27ac…",74.08404,2.146478,73.98,0.626099
"""H3K9ac-H3K9me3-H3K27ac""",73.55582,1.293899,73.96,1.123833
"""H3K4me3-H3K9me3-H3K27ac""",73.4632,1.999445,73.84,0.541295
…,…,…,…,…
"""H3K9ac""",71.9412,1.431197,71.46,1.532319
"""H3K9ac-H3K9me3""",71.9329,1.507742,71.3,1.6109
"""H3K9me3-H3K27me3""",50.7856,1.280173,52.7,0.821584


In [17]:
summary_df.write_csv(OUTPUT_PATH/ f"{DONOR}.csv", include_header=True)